# 5장 모델 자체의 도구 호출

에이전트가 도구를 호출해 외부 정보를 가져오고 작업을 수행합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 5장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1. 언어 모델 선택

도구 호출을 학습한 Gemma 4를 사용합니다. 이 모델은 프롬프트에 호출 형식을 직접 작성하지 않아도 API를 통해 도구를 요청할 수 있습니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B(내장 추론과 도구 호출 지원)
llm = LLM(model="gemma4:e4b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 모델이 생성하는 도구 호출

[기본 실습](chapter05.ipynb)에서 함수를 만들고 설명을 정의한 뒤, 모델이 도구를 선택하면 프로그램이 실행해 결과를 처리했습니다. 내장 도구 호출에서도 같은 단계가 필요하지만 호출 형식의 생성과 해석을 모델과 서버가 담당합니다.

![도구를 정의하고 실행하는 단계](../images/ch5_tools.png)

OpenAI 호환 엔드포인트로 요청을 보내 도구 호출이 어떤 형식으로 전달되는지 살펴봅니다. 도구로 사용할 함수는 앞 예제와 같습니다.

In [ ]:
def multiply(a: str, b: str) -> float:
    """Multiply two values."""
    return float(a) * float(b)

def get_weather(location: str) -> str:
    """Get the weather using a `location` parameter."""
    return f"Weather in {location}: Sunny, 72°F"

함수의 정의는 엔드포인트가 요구하는 JSON 형식으로 전달합니다. 아래는 `get_weather`의 이름과 설명, 인자 구조를 정의한 예입니다.

In [ ]:
tool_schema = {
  "type": "function",
  "function": {
    "name": "multiply",
    "description": "Multiply two numbers",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number"
        },
        "b": {
          "type": "number",
          "description": "Second number"
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  }
}

호환 엔드포인트는 이 JSON을 모델의 대화 템플릿에 맞게 변환하고 모델의 출력을 다시 도구 호출 형식으로 해석합니다. 다음 요청에서 이를 확인합니다.

In [ ]:
import json
import urllib.request
from rich import print

# 프롬프트
messages = [{"role": "user", "content": "What is 5.1 times 7.3?"}]

# 전송할 데이터 준비
body = {
    "model": "gemma4:e4b",
    "messages": messages,
    "stream": False,
    "tools": [tool_schema],
}

# 요청 준비
request = urllib.request.Request(
    "http://localhost:11434/api/chat",
    data=json.dumps(body).encode(),
    headers={"Content-Type": "application/json"},
)

# 요청을 보내고 응답 해석
with urllib.request.urlopen(request) as response:
    print(json.loads(response.read())["message"])

이전 예제에서는 모델에게 JSON 형식을 알려 주고 직접 해석했습니다. 이번에는 모델이 학습한 형식이 따로 있습니다. 서버가 도구 정의를 프롬프트에 넣고 응답을 다시 변환하는 과정을 살펴봅니다.

## 3. 도구 정의가 대화 템플릿으로 변환되는 과정

Gemma 4가 받는 시스템 메시지는 대략 다음 형식입니다.

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.8;">
<span style="color:#f97583;">&lt;|turn&gt;system</span>
<span style="color:#79b8ff;">&lt;|tool&gt;</span><span style="color:#85e89d;">declaration:</span><span style="background:#85e89d22; color:#85e89d; padding:1px 6px; border-radius:3px;">FUNCTION_NAME</span>{description:<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">DESCRIPTION</span><span style="color:#ffab70;">&lt;|"|&gt;</span>,parameters:{properties:{<span style="background:#85e89d22; color:#85e89d; padding:1px 6px; border-radius:3px;">PARAM_NAME</span>:{description:<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">PARAM_DESC</span><span style="color:#ffab70;">&lt;|"|&gt;</span>,type:<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">TYPE</span><span style="color:#ffab70;">&lt;|"|&gt;</span>}},required:[<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">PARAM_NAME</span><span style="color:#ffab70;">&lt;|"|&gt;</span>],type:<span style="color:#ffab70;">&lt;|"|&gt;</span>OBJECT<span style="color:#ffab70;">&lt;|"|&gt;</span>}}<span style="color:#79b8ff;">&lt;tool|&gt;</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
<span style="background:#D7D7D7; color:#000000; padding:1px 6px; border-radius:3px;">USER_PROMPT</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>

템플릿이 시스템 메시지에 도구 정보를 채웁니다. 이 순서와 특수 토큰은 모델이 도구 호출을 학습할 때 사용한 형식이므로 임의로 바꾸면 모델이 호출 형식을 따르기 어려울 수 있습니다. 다음은 날씨 도구의 JSON 정의입니다.

```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the weather using a location parameter.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "The city name"
        }
      },
      "required": ["location"]
    }
  }
}
```

Ollama는 이 정의를 읽어 모델이 기대하는 형식으로 변환합니다.

| JSON 입력 | Gemma 4 템플릿에 들어가는 값 |
|---|---|
| `"name": "get_weather"` | `declaration:get_weather{...}` |
| `"description": "Get the..."` | `description:<\|"\|>Get the...<\|"\|>` |
| `"type": "string"` | `type:<\|"\|>STRING<\|"\|>` |
| `"required": ["location"]` | `required:[<\|"\|>location<\|"\|>]` |
| 문자열 따옴표 `"..."` | 모델의 문자열 구분자 `<\|"\|>...<\|"\|>` |

날씨 도구와 파리 날씨를 묻는 사용자 질문을 채운 결과는 다음과 같습니다.

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.6;">
<span style="color:#f97583;">&lt;|turn&gt;system</span>
<span style="color:#79b8ff;">&lt;|tool&gt;</span><span style="color:#85e89d;">declaration:get_weather</span>{description:<span style="color:#ffab70;">&lt;|"|&gt;</span>Get the weather using a location parameter.<span style="color:#ffab70;">&lt;|"|&gt;</span>,parameters:{properties:{location:{description:<span style="color:#ffab70;">&lt;|"|&gt;</span>The city name<span style="color:#ffab70;">&lt;|"|&gt;</span>,type:<span style="color:#ffab70;">&lt;|"|&gt;</span>STRING<span style="color:#ffab70;">&lt;|"|&gt;</span>}},required:[<span style="color:#ffab70;">&lt;|"|&gt;</span>location<span style="color:#ffab70;">&lt;|"|&gt;</span>],type:<span style="color:#ffab70;">&lt;|"|&gt;</span>OBJECT<span style="color:#ffab70;">&lt;|"|&gt;</span>}}<span style="color:#79b8ff;">&lt;tool|&gt;</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
What is the weather in Paris?<span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>

이번에는 Ollama의 자동 템플릿 처리를 거치지 않고 모델에 직접 요청합니다. 먼저 위 템플릿대로 프롬프트를 작성합니다.

In [ ]:
# 모델이 기대하는 형식으로 대화 템플릿을 직접 구성
raw_prompt = """<|turn>system
<|tool>declaration:multiply{description:<|"|>Multiply two values `a` and `b`.<|"|>,parameters:{properties:{a:{type:<|"|>STRING<|"|>},b:{type:<|"|>STRING<|"|>}},required:[<|"|>a<|"|>,<|"|>b<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><turn|>
<|turn>user
What is 5.1 multiplied by 7.3?<turn|>
<|turn>model
"""

작성한 프롬프트를 Ollama의 `api/generate` 엔드포인트에 원문 그대로 전달합니다.

In [ ]:
import urllib.request
import json

# 요청 데이터 생성
data = json.dumps(
    {
        "model": llm.model,
        "prompt": raw_prompt,
        "raw": True,
        "stream": False
    }
).encode()

# 템플릿을 다시 적용하지 않고 전송
req = urllib.request.Request(
    "http://localhost:11434/api/generate",
    data=data,
    headers={"Content-Type": "application/json"}
)

# 모델의 원시 출력 확인
with urllib.request.urlopen(req) as response:
    print(json.loads(response.read()))

프롬프트 형식은 [Gemma 4의 대화 템플릿](https://huggingface.co/google/gemma-4-E4B-it/blob/main/chat_template.jinja)에 정의되어 있습니다. Ollama와 llama.cpp 등은 이 정의에 맞게 메시지를 변환합니다.

## 4. 함수 정의에서 JSON 스키마 만들기

API가 공통 JSON 스키마를 받으면 서버는 이를 모델별 대화 템플릿으로 바꿀 수 있습니다. 이제 파이썬의 `inspect`로 함수 시그니처를 읽고 자료형을 변환해 이 스키마를 만듭니다. 먼저 함수에서 어떤 정보를 얻을 수 있는지 확인합니다.

In [ ]:
import inspect

# 함수 정보 추출
name = get_weather.__name__  # 함수 이름
parameters = inspect.signature(get_weather).parameters  # 함수의 매개변수
docstring = inspect.getdoc(get_weather)  # 함수의 문서 문자열

# 함수 정보 표시
print(f"NAME:\n{name}")
print(f"PARAMETERS:\n{parameters}")
print(f"DOCSTRING:\n{docstring}")

시그니처에서 읽은 정보로 스키마를 만드는 함수를 작성합니다.

In [ ]:
import inspect
from typing import Callable

# 파이썬 자료형을 스키마의 자료형 이름으로 변환
TYPE_MAP = {
    str: "string", int: "integer", float: "number", 
    bool: "boolean", list: "array", dict: "object"
}

def tool_to_schema(function: Callable) -> dict:
    """파이썬 함수를 OpenAI 호환 도구 스키마로 변환한다."""
    signature = inspect.signature(function)

    # 함수 매개변수 정보 추출
    properties, required = {}, []
    for name, parameter in signature.parameters.items():
        properties[name] = {"type": TYPE_MAP.get(parameter.annotation, "string")}
        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    # 스키마 구성
    schema = {
        "type": "function",
        "function": {
            "name": function.__name__,
            "description": inspect.getdoc(function),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }

    return schema

이 함수로 만든 스키마를 앞에서 직접 작성한 것과 비교합니다.

In [ ]:
print(tool_to_schema(multiply))

## 5. `NativeTools` 구현

도구를 스키마로 바꿀 수 있으므로 2장에서 만든 `LLM` 클래스의 `tools` 매개변수로 전달할 수 있습니다. `NativeTools` 클래스가 등록한 도구와 모델의 도구 호출 응답을 관리합니다.

In [ ]:
import json
from illustrated_agents.chapters.ch2 import Response
from illustrated_agents.chapters.ch5 import Tools


class NativeTools(Tools):
    """모델 자체의 함수 호출을 지원하는 도구 목록이다."""

    @property
    def schemas(self) -> list[dict]:
        """모델에 전달할 도구의 함수 스키마를 반환한다."""
        return [
            tool_to_schema(tool["function"]) for tool in self.registry.values()
        ]

    @property
    def prompt(self) -> str:
        """내장 도구 호출에는 별도 프롬프트가 필요하지 않아 빈 문자열을 반환한다."""
        return ""

    def parse(self, response: Response) -> Response:
        """도구 호출을 해석한다."""
        # 도구 호출이 없으면 받은 응답을 그대로 반환
        if not response.tool_call:
            return response

        # 도구 호출에서 이름과 인자 추출
        args = response.tool_call["function"]["arguments"]
        if isinstance(args, str):
            args = json.loads(args)
        tool_call = {
            "tool": response.tool_call["function"]["name"],
            "kwargs": args,
        }

        # 응답에 해석한 도구 호출 추가
        return Response(
            content=response.content,
            reasoning=response.reasoning,
            tool_call=tool_call,
        )

    def observation(self, result: str) -> tuple[str, str]:
        """내장 도구 호출의 결과는 'tool' 역할로 전달한다."""
        return "tool", str(result)

    def is_done(self, response: Response) -> bool:
        """도구 호출이 없으면 TinyAgent가 작업을 마친다."""
        return not response.tool_call

함수와 설명을 도구 목록에 등록합니다.

In [ ]:
# 도구 등록
tools = NativeTools()
tools.add_tool("multiply", multiply)
tools.add_tool("get_weather", get_weather)

`schemas`로 등록한 도구의 JSON 스키마를 확인합니다.

In [ ]:
tools.schemas

서버가 도구 설명을 JSON에서 읽어 대화 템플릿에 넣으므로 `tools.prompt`에 호출 지침을 다시 작성할 필요가 없습니다.

## 6. `TinyAgent` 실행

`NativeTools`를 전달해 에이전트를 초기화합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import TinyAgent

# 도구
tools = NativeTools()
tools.add_tool("multiply", multiply)
tools.add_tool("get_weather", get_weather)

# 메모리
memory = Memory()

# 에이전트 초기화
agent = TinyAgent(llm=llm, memory=memory, tools=tools)

도구가 필요한 질문을 보내 모델이 올바른 도구를 선택하는지 봅니다.

예제 질문은 “5.1 곱하기 7.3은 얼마인가요?”입니다.

In [ ]:
agent.run("What is 5.1 times 7.3?")

저자의 예에서는 도구를 호출했습니다. 다른 결과가 나오면 실행 기록을 열어 선택된 도구와 인자를 확인합니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

도구를 사용하지 않고 답할 수 있는 질문도 보냅니다.

이번 질문은 “홍학에 대해 두 문장으로 알려 주세요”입니다.

In [ ]:
agent.run("Hi! Tell me something about flamingos in two sentences.")

모델은 직접 답할지 도구를 호출할지 결정할 수 있지만 항상 적절한 선택을 하는 것은 아닙니다. 도구가 수백 개면 설명이 컨텍스트 창을 많이 차지하고 선택도 어려워집니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

메모리에 남은 대화와 도구 결과를 확인합니다.

In [ ]:
from rich import print
print(agent.memory.get_messages())

---

# 구현한 내용

프롬프트에 JSON이나 XML 해석 지침을 쓰는 대신 모델의 내장 도구 호출을 사용했습니다. `NativeTools`는 호출 정보를 처리합니다. `LLM`, `Memory`, `TinyAgent`는 프롬프트 방식과 내장 방식에서 필요한 정보를 전달합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import what_we_built_native; what_we_built_native

# 다음 실습

[스킬](chapter05_skills.ipynb)에서는 에이전트가 작업별 지침을 읽도록 합니다. [6장](../chapter06/chapter06.ipynb)에서는 추론과 행동을 반복하는 ReAct를 구현합니다.